# Data Science & Database Assignment: Session 2
## Topic: Database Setup + CREATE TABLE

**Student Name:** Atharv Chaudhari  
**Course:** Data Science & Database Fundamentals  
**Database Name:** `foodie_app`  
**Core Concepts:** RDBMS Installation, DDL (`CREATE DATABASE`, `CREATE TABLE`), Primary Keys, Constraints (`UNIQUE`, `NOT NULL`), Data Type Selection, Syntax Error Debugging  
**Status:** Completed & Pre-Executed  

---

### Core Learning Objectives
1. Set up and verify database server connections (MySQL Community Server / SQLite).
2. Design and implement schemas for real-world e-commerce & food delivery architectures (Zomato & Flipkart).
3. Select appropriate database data types (`VARCHAR`, `INT`, `DECIMAL`, `TIMESTAMP`).
4. Apply relational integrity constraints (`PRIMARY KEY`, `UNIQUE`, `NOT NULL`, `DEFAULT`).
5. Practice error diagnostics by intentionally creating, inspecting, and resolving DDL syntax errors.


## Task 1 & 2: Database Setup & Connection (`foodie_app`)
> **Problem Statement 1:** Install MySQL Community Server or SQLite on your system and verify the installation by connecting to the database using the command line or a GUI tool like MySQL Workbench or DB Browser for SQLite.  
> **Problem Statement 2:** Create a new database named `'foodie_app'` to simulate a Zomato-style backend.

### 1. Installation & Verification Methods

| Engine | Command Line (CLI) | Graphical User Interface (GUI) |
| :--- | :--- | :--- |
| **MySQL Community Server** | `mysql -u root -p` | **MySQL Workbench** / DBeaver |
| **SQLite (Built-in)** | `sqlite3 foodie_app.db` | **DB Browser for SQLite** |

### 2. DDL Commands
```sql
-- In MySQL:
CREATE DATABASE IF NOT EXISTS foodie_app;
USE foodie_app;

-- In SQLite:
-- Created automatically on first connection:
-- sqlite3 foodie_app.db
```


In [1]:
import sqlite3
import pandas as pd

# Connect to SQLite database 'foodie_app.db'
conn = sqlite3.connect("foodie_app.db")
cursor = conn.cursor()

print("[SUCCESS] Successfully connected to 'foodie_app.db' database engine.")


[SUCCESS] Successfully connected to 'foodie_app.db' database engine.


## Task 3: Create `restaurants` Table (Zomato-Style Backend)
> **Problem Statement:** Write a `CREATE TABLE` statement to define a `'restaurants'` table in the `'foodie_app'` database with the following columns: `id` (integer, primary key), `name` (varchar/character, max 100), `cuisine` (varchar/character, max 50), `rating` (decimal, e.g., 4.5), and `location` (varchar/character, max 100).

### Schema Specification
| Column Name | Data Type | Constraints | Description |
| :--- | :--- | :--- | :--- |
| `id` | `INT` / `INTEGER` | `PRIMARY KEY` | Unique restaurant identifier |
| `name` | `VARCHAR(100)` | `NOT NULL` | Restaurant brand name |
| `cuisine` | `VARCHAR(50)` | `NOT NULL` | Culinary category (e.g. North Indian, Italian) |
| `rating` | `DECIMAL(2, 1)` | `NOT NULL` | Average star rating (e.g. 4.5 out of 5.0) |
| `location` | `VARCHAR(100)` | `NOT NULL` | City neighborhood / address |

### SQL Statement:
```sql
CREATE TABLE restaurants (
    id INT PRIMARY KEY,
    name VARCHAR(100) NOT NULL,
    cuisine VARCHAR(50) NOT NULL,
    rating DECIMAL(2, 1) NOT NULL,
    location VARCHAR(100) NOT NULL
);
```


In [2]:
# Drop existing table for idempotent execution
cursor.execute("DROP TABLE IF EXISTS restaurants;")

create_restaurants_sql = """
CREATE TABLE restaurants (
    id INTEGER PRIMARY KEY,
    name VARCHAR(100) NOT NULL,
    cuisine VARCHAR(50) NOT NULL,
    rating DECIMAL(2, 1) NOT NULL,
    location VARCHAR(100) NOT NULL
);
"""
cursor.execute(create_restaurants_sql)

# Insert sample restaurants to verify data storage
sample_restaurants = [
    (1, 'Punjab Grill', 'North Indian', 4.5, 'Connaught Place, New Delhi'),
    (2, 'Biryani Blues', 'Hyderabadi', 4.3, 'Indiranagar, Bengaluru'),
    (3, 'Bastian', 'Seafood / Continental', 4.7, 'Bandra West, Mumbai')
]
cursor.executemany("INSERT INTO restaurants VALUES (?, ?, ?, ?, ?);", sample_restaurants)
conn.commit()

df_restaurants = pd.read_sql_query("SELECT * FROM restaurants;", conn)
print("Task 3 Output: Populated 'restaurants' Table")
display(df_restaurants)


Task 3 Output: Populated 'restaurants' Table


,id,name,cuisine,rating,location
0,1,Punjab Grill,North Indian,4.5,"Connaught Place, New Delhi"
1,2,Biryani Blues,Hyderabadi,4.3,"Indiranagar, Bengaluru"
2,3,Bastian,Seafood / Continental,4.7,"Bandra West, Mumbai"


## Task 4: Design & Create `users` Table (Flipkart-Style App)
> **Problem Statement:** Design and create a `'users'` table for a Flipkart-style app with columns: `user_id` (primary key), `username`, `email`, `phone_number`, and `created_at` (date/time). Pick appropriate data types for each column.  
> *Hint: Think about which columns should be unique and which data types best fit email and phone numbers.*

### Architectural Data Type Decisions

1. **`phone_number` as `VARCHAR(15)` instead of `INTEGER`:**
   - Phone numbers can begin with leading zeros (e.g., `011...`). An integer column would truncate leading zeros.
   - International standards (E.164) include country codes and symbols like `+` (e.g. `+919876543210`).
   - We never perform mathematical operations (addition, averaging) on telephone numbers.
2. **`email` with `UNIQUE` Constraint:**
   - Each customer account on an e-commerce platform must be associated with a single distinct identity.
3. **`created_at` with `TIMESTAMP DEFAULT CURRENT_TIMESTAMP`:**
   - Automatically timestamps user registration in UTC without manual client-side input.

### SQL Statement:
```sql
CREATE TABLE users (
    user_id INT PRIMARY KEY,
    username VARCHAR(50) NOT NULL,
    email VARCHAR(100) UNIQUE NOT NULL,
    phone_number VARCHAR(15) UNIQUE NOT NULL,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
```


In [3]:
# Drop existing users table
cursor.execute("DROP TABLE IF EXISTS users;")

create_users_sql = """
CREATE TABLE users (
    user_id INTEGER PRIMARY KEY,
    username VARCHAR(50) NOT NULL,
    email VARCHAR(100) UNIQUE NOT NULL,
    phone_number VARCHAR(15) UNIQUE NOT NULL,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
"""
cursor.execute(create_users_sql)

# Insert sample Flipkart user accounts
sample_users = [
    (101, 'atharv_c', 'atharv@example.com', '+919876543210'),
    (102, 'priya_sharma', 'priya.s@example.com', '+919812345678'),
    (103, 'rahul_verma', 'rahul.v@example.com', '+919898765432')
]
cursor.executemany("""
    INSERT INTO users (user_id, username, email, phone_number)
    VALUES (?, ?, ?, ?);
""", sample_users)
conn.commit()

df_users = pd.read_sql_query("SELECT * FROM users;", conn)
print("Task 4 Output: Populated 'users' Table with Constraints")
display(df_users)


Task 4 Output: Populated 'users' Table with Constraints


,user_id,username,email,phone_number,created_at
0,101,atharv_c,atharv@example.com,+919876543210,2026-10-27 07:18:20
1,102,priya_sharma,priya.s@example.com,+919812345678,2026-10-27 07:18:20
2,103,rahul_verma,rahul.v@example.com,+919898765432,2026-10-27 07:18:20


## Task 5: Intentional Syntax Mistake & Error Diagnostics
> **Problem Statement:** Intentionally make a mistake in your `CREATE TABLE` statement (such as missing a comma or using an unsupported data type), run it, and then fix the error based on the message you receive.  
> *Hint: Take a screenshot of the error and the corrected SQL statement for your records.*

---

### 1. Intentional Bug Injection: Missing Comma
We deliberately omit the comma `,` between the `email` column definition and the `phone` column definition:

```sql
CREATE TABLE invalid_demo_table (
    id INT PRIMARY KEY,
    username VARCHAR(50),
    email VARCHAR(100)       -- <-- MISSING COMMA HERE!
    phone VARCHAR(15)
);
```

### 2. Error Message Diagnostic
When the database parser processes `email VARCHAR(100)`, it expects either a comma `,` to separate the next column or a closing parenthesis `)` to finish the table definition. Encountering the identifier `phone` triggers an immediate parser failure:
* **MySQL:** `ERROR 1064 (42000): You have an error in your SQL syntax near 'phone VARCHAR(15))' at line 5`
* **SQLite:** `OperationalError: near "phone": syntax error`

### 3. The Fix
Add the comma `,` following `VARCHAR(100)`:
```sql
CREATE TABLE fixed_demo_table (
    id INT PRIMARY KEY,
    username VARCHAR(50),
    email VARCHAR(100),      -- <-- FIXED WITH COMMA
    phone VARCHAR(15)
);
```


In [4]:
# 1. Deliberate Error Demonstration
invalid_sql = """
CREATE TABLE invalid_demo_table (
    id INT PRIMARY KEY,
    username VARCHAR(50),
    email VARCHAR(100)
    phone VARCHAR(15)
);
"""

print("--- Running Invalid SQL Statement ---")
try:
    cursor.execute(invalid_sql)
except sqlite3.OperationalError as err:
    print(f"[CAUGHT EXPECTED ERROR]: {type(err).__name__}: {err}")
    print("Diagnosis: SQL syntax error near 'phone' because the previous column definition lacked a delimiter comma.")

# 2. Executing Fixed SQL Statement
cursor.execute("DROP TABLE IF EXISTS fixed_demo_table;")
fixed_sql = """
CREATE TABLE fixed_demo_table (
    id INT PRIMARY KEY,
    username VARCHAR(50),
    email VARCHAR(100),
    phone VARCHAR(15)
);
"""

print("\n--- Running Corrected SQL Statement ---")
cursor.execute(fixed_sql)
conn.commit()
print("[SUCCESS] Table 'fixed_demo_table' created successfully!")

# Clean up connection
conn.close()


--- Running Invalid SQL Statement ---
[CAUGHT EXPECTED ERROR]: OperationalError: near "phone": syntax error
Diagnosis: SQL syntax error near 'phone' because the previous column definition lacked a delimiter comma.

--- Running Corrected SQL Statement ---
[SUCCESS] Table 'fixed_demo_table' created successfully!


---

## 📌 Summary of Core Concepts Mastered

| Concept | SQL Syntax / Keyword | Importance |
| :--- | :--- | :--- |
| **Database Creation** | `CREATE DATABASE db_name;` | Creates the logical container for tables |
| **Table Creation** | `CREATE TABLE tbl (col type);` | Defines the structured schema and column types |
| **Primary Key** | `PRIMARY KEY` | Enforces entity integrity; guarantees row uniqueness |
| **Unique Constraint** | `UNIQUE` | Prevents duplicate entries (essential for emails & phones) |
| **Not Null** | `NOT NULL` | Mandates that a column must have a valid value |
| **Default Timestamps**| `DEFAULT CURRENT_TIMESTAMP` | Automates audit logging upon row insertion |
